# Analyse Exploratoire Enrichie des Donnees Clients de Cartes de Credit
## Comprehension Approfondie de la Structure du Dataset

## 1. Chargement et Inspection Initiale des Donnees

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Charger le fichier de donnees
df = pd.read_csv("default_of_credit_card_clients.csv")

print("Affichage des premieres lignes du dataset:")
print(df.head())

Affichage des premieres lignes du dataset:
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE   AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   3    90000.0    2          2         2  34.0      0      0      0      0   
1   4    50000.0    2          2         1  37.0      0      0      0      0   
2   5    50000.0    1          2         1  57.0     -1      0     -1      0   
3   6    50000.0    1          1         2  37.0      0      0      0      0   
4   7        NaN    1          1         2  29.0      0      0      0      0   

   ...  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  \
0  ...      15549    1518.0      1500      1000      1000      1000      5000   
1  ...      29547    2000.0      2019      1200      1100      1069      1000   
2  ...      19131    2000.0     36681     10000      9000       689       679   
3  ...      20024    2500.0      1815       657      1000      1000       800   
4  ...     473944   55000.0     40000     38000     20239     13750    

### 1.1 Dimensions et Structure du Dataset

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_palette("husl")

print("Bibliotheques de graphiques chargees avec succes")

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [ ]:
print(f"Nombre de lignes: {df.shape[0]}")
print(f"Nombre de colonnes: {df.shape[1]}")
print(f"\nMémoire utilisée: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

### 1.2 Types de Donnees et Informations Detaillees

In [ ]:
# Informations detaillees sur les colonnes
info_dict = {
    'Colonne': df.columns,
    'Type': df.dtypes,
    'Non-Null': df.count(),
    'Null': df.isnull().sum(),
    'Unique': df.nunique()
}

info_df = pd.DataFrame(info_dict)
print(info_df)

## 2. Analyse des Valeurs Manquantes

### 2.1 Detection des Valeurs Manquantes

In [ ]:
# Compter les valeurs manquantes
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Colonne': missing_count.index,
    'Nombre_Manquant': missing_count.values,
    'Pourcentage': missing_percent.values
}).sort_values('Nombre_Manquant', ascending=False)

missing_df = missing_df[missing_df['Nombre_Manquant'] > 0]

if len(missing_df) > 0:
    print("Colonnes avec valeurs manquantes:")
    print(missing_df)
else:
    print("Aucune valeur manquante detectée!")

### 2.2 Visualisation des Valeurs Manquantes

In [ ]:
if missing_percent.sum() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graphique en barres des valeurs manquantes
    missing_percent[missing_percent > 0].plot(kind='barh', ax=axes[0], color='salmon')
    axes[0].set_xlabel('Pourcentage de donnees manquantes')
    axes[0].set_title('Pourcentage de donnees manquantes par colonne')
    
    # Graphique circulaire du total
    total_missing = missing_percent.sum()
    axes[1].pie([100 - total_missing, total_missing], 
                labels=['Donnees Valides', 'Donnees Manquantes'],
                autopct='%1.1f%%',
                colors=['lightgreen', 'salmon'])
    axes[1].set_title('Proportion globale de donnees valides vs manquantes')
    
    plt.tight_layout()
    plt.show()
else:
    print("Pas de valeurs manquantes - pas de graphique a afficher")

## 3. Analyse des Valeurs Aberrantes (Outliers)

### 3.1 Selection des Colonnes Numeriques

In [ ]:
# Selectionner les colonnes numeriques
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Colonnes numeriques: {len(num_cols)}")
print(f"Colonnes categoriques: {len(cat_cols)}")
print(f"\nColonnes numeriques: {num_cols}")
print(f"\nColonnes categoriques: {cat_cols}")

### 3.2 Boxplots de toutes les Variables Numeriques

In [ ]:
# Boxplot de toutes les variables numeriques
plt.figure(figsize=(20, 10))
sns.boxplot(data=df[num_cols])
plt.title("Boxplots de toutes les variables numeriques", fontsize=16, fontweight='bold')
plt.xlabel("Variables")
plt.ylabel("Valeurs")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.3 Detection Statistique des Outliers par la Methode IQR

In [ ]:
# Calcul des outliers avec la methode IQR
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    outlier_summary[col] = {
        'Nombre_Outliers': len(outliers),
        'Pourcentage': (len(outliers) / len(df)) * 100,
        'Borne_Inferieure': lower_bound,
        'Borne_Superieure': upper_bound
    }

outlier_df = pd.DataFrame(outlier_summary).T.sort_values('Pourcentage', ascending=False)
print("Resume des outliers par colonne:")
print(outlier_df)

### 3.4 Visualisation des Outliers

In [ ]:
# Graphique des pourcentages d'outliers
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Graphique en barres du nombre d'outliers
outlier_counts = pd.Series({col: outlier_summary[col]['Nombre_Outliers'] for col in num_cols})
outlier_counts.sort_values(ascending=False).plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_xlabel('Nombre de donnees aberrantes')
axes[0].set_title('Nombre de donnees aberrantes par variable')
axes[0].grid(axis='x', alpha=0.3)

# Graphique en barres du pourcentage d'outliers
outlier_pcts = pd.Series({col: outlier_summary[col]['Pourcentage'] for col in num_cols})
outlier_pcts.sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='skyblue')
axes[1].set_xlabel('Pourcentage de donnees aberrantes')
axes[1].set_title('Pourcentage de donnees aberrantes par variable')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Analyse Statistique Descriptive

### 4.1 Statistiques Descriptives des Variables Numeriques

In [ ]:
# Statistiques descriptives completes
stats_df = df[num_cols].describe().T
stats_df['Skewness'] = df[num_cols].skew()
stats_df['Kurtosis'] = df[num_cols].kurtosis()

print("Statistiques descriptives des variables numeriques:")
print(stats_df.round(3))

### 4.2 Histogrammes de Distribution

In [ ]:
# Histogrammes des distributions
fig = plt.figure(figsize=(20, 16))

num_vars = len(num_cols)
cols_per_row = 4
rows = (num_vars + cols_per_row - 1) // cols_per_row

for idx, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols_per_row, idx)
    plt.hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    plt.title(f'Distribution de {col}', fontweight='bold')
    plt.xlabel('Valeur')
    plt.ylabel('Frequence')
    plt.grid(axis='y', alpha=0.3)

plt.suptitle('Distributions de toutes les variables numeriques', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### 4.3 Distributions avec Courbe de Densite

In [ ]:
# Distributions avec courbes de densite pour les principales variables
main_vars = [col for col in num_cols if col != 'default_payment_next_month'][:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(main_vars):
    axes[idx].hist(df[col].dropna(), bins=50, density=True, alpha=0.6, color='skyblue', edgecolor='black')
    df[col].dropna().plot(kind='kde', ax=axes[idx], color='red', linewidth=2)
    axes[idx].set_title(f'{col}')
    axes[idx].set_xlabel('Valeur')
    axes[idx].set_ylabel('Densite')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Distributions avec courbes de densite', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Analyse de la Variable Cible

### 5.1 Desequilibre de Classes

In [ ]:
# Analyser la variable cible
target_col = 'default_payment_next_month'

print("Distribution brute de la classe cible:")
print(df[target_col].value_counts())
print("\nDistribution en pourcentage:")
print(df[target_col].value_counts(normalize=True) * 100)

### 5.2 Visualisation du Desequilibre

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Graphique en barres
target_counts = df[target_col].value_counts()
colors = ['lightgreen', 'salmon']
target_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Nombre de clients par classe', fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre')
axes[0].set_xticklabels(['Non-Defaut (0)', 'Defaut (1)'], rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Graphique circulaire
target_pct = df[target_col].value_counts(normalize=True) * 100
axes[1].pie(target_pct.values, 
            labels=['Non-Defaut', 'Defaut'],
            autopct='%1.1f%%',
            colors=colors,
            startangle=90)
axes[1].set_title('Proportion des classes', fontweight='bold')

# Graphique de ratio
ratio = target_counts[0] / target_counts[1]
axes[2].barh(['Ratio Non-Defaut/Defaut'], [ratio], color='steelblue')
axes[2].set_xlabel('Ratio')
axes[2].set_title(f'Ratio de desequilibre: 1:{ratio:.2f}', fontweight='bold')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInformation importante: Le dataset contient un desequilibre de classes.")
print(f"Ratio: {ratio:.2f} clients non-defaut pour 1 client en defaut")

## 6. Analyse des Variables Categoriques

### 6.1 Distribution des Variables Categoriques

In [ ]:
# Variables categoriques communes
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE']

print("Distribution des variables categoriques:")
for col in categorical_features:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().sort_index())
        print(f"Pourcentage:")
        print((df[col].value_counts(normalize=True).sort_index() * 100).round(2))

### 6.2 Visualisation des Variables Categoriques

In [ ]:
# Visualiser les distributions des variables categoriques
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cat_to_plot = [col for col in categorical_features if col in df.columns]

for idx, col in enumerate(cat_to_plot):
    ax = axes[idx]
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=ax, color='skyblue')
    ax.set_title(f'Distribution de {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Nombre de clients')
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# Masquer les axes inutiles
for idx in range(len(cat_to_plot), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Correlations et Multicollinearite

### 7.1 Matrice de Correlation Complete

In [ ]:
# Calculer la matrice de correlation
corr_matrix = df[num_cols].corr()

# Afficher la matrice
plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, 
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})

plt.title("Matrice de correlation de toutes les variables numeriques", fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 7.2 Identification des Correlations Fortes

In [ ]:
# Trouver les paires de variables fortement correlees
print("Paires de variables avec une correlation absolue > 0.8:")
print("="*60)

strong_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.8:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            strong_corr_pairs.append((col1, col2, corr_value))
            print(f"{col1} <-> {col2}: {corr_value:.4f}")

if len(strong_corr_pairs) == 0:
    print("Aucune paire avec correlation absolue > 0.8 trouvee")

print(f"\nTotal de paires fortement correlees: {len(strong_corr_pairs)}")

### 7.3 Correlation avec la Variable Cible

In [ ]:
if target_col in df.columns:
    # Correlations avec la variable cible
    target_corr = df[num_cols].corr()[target_col].drop(target_col).sort_values(ascending=False)
    
    print("Correlation des variables avec la cible:")
    print(target_corr)
    
    # Visualiser les correlations avec la cible
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Graphique 1: Barres positives et negatives
    colors = ['green' if x > 0 else 'red' for x in target_corr.values]
    target_corr.plot(kind='barh', ax=axes[0], color=colors)
    axes[0].set_xlabel('Correlation')
    axes[0].set_title('Correlation de chaque variable avec la cible', fontweight='bold')
    axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[0].grid(axis='x', alpha=0.3)
    
    # Graphique 2: Valeurs absolues triees
    abs_corr = target_corr.abs().sort_values(ascending=False)
    abs_corr.plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_xlabel('Correlation absolue')
    axes[1].set_title('Valeurs absolues triees - Impact sur la cible', fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Analyse de Covariance et Relations Bivariees

### 8.1 Scatterplots des Variables Principales

In [ ]:
# Selectionner les variables les plus importantes
important_vars = target_corr.abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(important_vars):
    ax = axes[idx]
    
    # Scatterplot avec couleurs par classe
    for class_label in df[target_col].unique():
        mask = df[target_col] == class_label
        label = f"Defaut" if class_label == 1 else "Non-Defaut"
        ax.scatter(df[mask].index, df[mask][col], 
                  alpha=0.5, label=label, s=20)
    
    ax.set_title(f'{col} vs Classe (Correlation: {target_corr[col]:.3f})', fontweight='bold')
    ax.set_xlabel('Index')
    ax.set_ylabel(col)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Variables principales vs Classe Cible', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Analyse des Tendances Temporelles et Patterns

### 9.1 Distribution par Classe pour Variables Continues

In [ ]:
# Comparer les distributions par classe
top_vars = target_corr.abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(top_vars):
    ax = axes[idx]
    
    for class_label in sorted(df[target_col].unique()):
        data = df[df[target_col] == class_label][col].dropna()
        label = f"Defaut" if class_label == 1 else "Non-Defaut"
        ax.hist(data, bins=40, alpha=0.6, label=label)
    
    ax.set_title(f'Distribution de {col} par classe', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequence')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distributions comparees par classe cible', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.2 Boxplots Compares par Classe

In [ ]:
# Boxplots compares
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(top_vars):
    ax = axes[idx]
    
    # Preparer les donnees pour le boxplot
    data_to_plot = [df[df[target_col] == 0][col].dropna(),
                    df[df[target_col] == 1][col].dropna()]
    
    bp = ax.boxplot(data_to_plot, labels=['Non-Defaut', 'Defaut'],
                     patch_artist=True)
    
    # Colorer les boites
    colors = ['lightgreen', 'salmon']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_title(f'Boxplot de {col}', fontweight='bold')
    ax.set_ylabel(col)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparaison des distributions par classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Analyse Resumee et Conclusions

### 10.1 Resume des Trouvailles Principales

In [ ]:
print("="*70)
print("RESUME DE L'ANALYSE EXPLORATOIRE")
print("="*70)

print(f"\n1. STRUCTURE DU DATASET:")
print(f"   Nombre total d'observations: {df.shape[0]:,}")
print(f"   Nombre de variables: {df.shape[1]}")
print(f"   Variables numeriques: {len(num_cols)}")
print(f"   Variables categoriques: {len(cat_cols)}")

print(f"\n2. VALEURS MANQUANTES:")
missing_total = df.isnull().sum().sum()
print(f"   Total de valeurs manquantes: {missing_total}")
if missing_total == 0:
    print("   Conclusion: Aucune valeur manquante detectée")
else:
    print(f"   Pourcentage de donnees manquantes: {(missing_total / (df.shape[0] * df.shape[1]) * 100):.2f}%")

print(f"\n3. VALEURS ABERRANTES:")
total_outliers = sum(outlier_summary[col]['Nombre_Outliers'] for col in num_cols)
print(f"   Total de donnees aberrantes detectees: {total_outliers:,}")
print(f"   Variables les plus affectees: {outlier_df.head(3).index.tolist()}")

print(f"\n4. CLASSE CIBLE:")
print(f"   Classe 0 (Non-Defaut): {(df[target_col] == 0).sum():,} clients ({(df[target_col] == 0).sum()/len(df)*100:.1f}%)")
print(f"   Classe 1 (Defaut): {(df[target_col] == 1).sum():,} clients ({(df[target_col] == 1).sum()/len(df)*100:.1f}%)")
print(f"   Desequilibre: Ratio de {ratio:.1f}:1")

print(f"\n5. VARIABLES LES PLUS CORRELEES AVEC LA CIBLE:")
for i, (var, corr_val) in enumerate(target_corr.abs().nlargest(5).items(), 1):
    actual_corr = target_corr[var]
    print(f"   {i}. {var}: {actual_corr:.4f}")

print(f"\n6. MULTICOLLINEARITE:")
print(f"   Nombre de paires avec correlation > 0.8: {len(strong_corr_pairs)}")
if len(strong_corr_pairs) > 0:
    print("   Paires principales:")
    for col1, col2, corr in strong_corr_pairs[:3]:
        print(f"      - {col1} <-> {col2}: {corr:.4f}")
else:
    print("   Pas de probleme majeur de multicollinearite")

print("\n" + "="*70)

### 10.2 Recommendations pour le Nettoyage des Donnees

In [ ]:
recommendations = f"""
RECOMMANDATIONS POUR LE NETTOYAGE ET LA PREPARATION:

1. TRAITEMENT DES DONNEES MANQUANTES:
   - Aucune action requise si aucune donnee manquante
   - Sinon, utiliser l'imputation par la mediane pour les variables numeriques
   
2. TRAITEMENT DES VALEURS ABERRANTES:
   - Conserver les outliers (donnees financieres legitimes)
   - Appliquer une standardisation robuste (RobustScaler) qui est resistante aux valeurs extremes
   - Envisager une transformation logarithmique pour les variables tres asymetriques
   
3. GESTION DU DESEQUILIBRE DE CLASSES:
   - Appliquer SMOTE sur l'ensemble d'entraînement uniquement
   - Utiliser class_weight='balanced' dans les modeles de classification
   - Utiliser des metriques appropriees: ROC-AUC et Recall, pas seulement Accuracy
   
4. ENCODAGE DES VARIABLES CATEGORIQUES:
   - Appliquer OneHotEncoder avec drop='first' pour eviter la multicollinearite
   
5. SELECTION DE VARIABLES:
   - Utiliser SelectKBest ou mutual_info_classif pour reduire la dimensionalite
   - Tenir compte des correlations fortes identifiees
   
6. NORMALISATION:
   - Appliquer RobustScaler pour les variables numeriques
   - Eviter StandardScaler qui est sensible aux outliers
"""

print(recommendations)

## 11. Pret pour les Etapes Suivantes

In [ ]:
print("Etapes suivantes:")
print("")
print("1. Nettoyage et transformation des donnees")
print("2. Selection et ingenierie des caracteristiques")
print("3. Division des donnees train/test")
print("4. Equilibrage des classes (SMOTE)")
print("5. Normalisation/standardisation")
print("6. Entrainement de modeles de classification")
print("7. Evaluation et comparaison des modeles")
print("8. Optimisation des hyperparametres")
print("9. Validation finale et interpretabilite")